# Table Lookup Model

Esempio didattico, semplice e commentato.

## Obiettivo

Un **Table Lookup Model** memorizza direttamente ciò che è stato osservato:

- stato corrente;
- azione eseguita;
- stato successivo;
- ricompensa ricevuta.

Quando il modello viene interrogato con una coppia `(stato, azione)`, restituisce la media delle transizioni osservate.

In [28]:
from collections import defaultdict
import random

# Definizione funzioni di transizione reali (che dovranno essere apprese dal modello)

In [29]:
ACTIONS = ["left", "right"]
STATES = ["A", "B", "C", "D"]


# Definizione in modo casuale delle probabilità di transizione per ogni coppia (stato, azione) verso un nuovo stato con associata media e varianza di una gaussiana per la ricompensa. Queste probabilità sono sconosciute all'agente e devono essere apprese dal modello.
TRANSITION_PROBABILITIES = {}

for state in STATES:
    for action in ACTIONS:
        next_states = random.sample(STATES, k=random.randint(1, len(STATES)))
        probabilities = [random.random() for _ in next_states]
        total = sum(probabilities)
        probabilities = [p / total for p in probabilities]
        reward_mean = random.uniform(-1, 1)
        reward_variance = random.uniform(0.1, 1)
        TRANSITION_PROBABILITIES[(state, action)] = {
            "next_states": next_states,
            "probabilities": probabilities,
            "reward_mean": reward_mean,
            "reward_variance": reward_variance,
        }
        
# Pretty print
for (state, action), data in TRANSITION_PROBABILITIES.items():
    print(f"Stato: {state}, Azione: {action}")
    for next_state, prob in zip(data["next_states"], data["probabilities"]):
        print(f"  Stato successivo: {next_state}, Probabilità: {prob:.2f}")
    print(f"  Media ricompensa: {data['reward_mean']:.2f}, Varianza ricompensa: {data['reward_variance']:.2f}")

Stato: A, Azione: left
  Stato successivo: A, Probabilità: 0.55
  Stato successivo: B, Probabilità: 0.45
  Media ricompensa: -0.93, Varianza ricompensa: 1.00
Stato: A, Azione: right
  Stato successivo: A, Probabilità: 0.47
  Stato successivo: D, Probabilità: 0.51
  Stato successivo: B, Probabilità: 0.02
  Media ricompensa: 0.62, Varianza ricompensa: 0.28
Stato: B, Azione: left
  Stato successivo: B, Probabilità: 0.17
  Stato successivo: A, Probabilità: 0.45
  Stato successivo: C, Probabilità: 0.21
  Stato successivo: D, Probabilità: 0.18
  Media ricompensa: 0.73, Varianza ricompensa: 0.67
Stato: B, Azione: right
  Stato successivo: A, Probabilità: 0.48
  Stato successivo: B, Probabilità: 0.26
  Stato successivo: D, Probabilità: 0.25
  Media ricompensa: -0.89, Varianza ricompensa: 0.29
Stato: C, Azione: left
  Stato successivo: C, Probabilità: 0.09
  Stato successivo: B, Probabilità: 0.81
  Stato successivo: D, Probabilità: 0.10
  Media ricompensa: -0.79, Varianza ricompensa: 0.42
Stato

In [30]:
class TableLookupModel:
    """Modello deterministico basato su una tabella di transizioni."""

    def __init__(self):
        # Chiave: (stato, azione)
        # Valore: (stato_successivo, ricompensa)
        self.table = defaultdict(list)

    def update(self, state, action, next_state, reward):
        """Memorizza o aggiorna una transizione osservata."""
        self.table[(state, action)].append((next_state, reward))

    def predict(self, state, action):
        """Restituisce la transizione nota per la coppia stato-azione."""
        key = (state, action)
    
        if key not in self.table:
            raise KeyError(f"Transizione non conosciuta: {key}")
        
        output = defaultdict(lambda: [0.0, 0.0])  # stato_successivo -> [somma_visite, somma_ricompensa]
        # Restituisce le coppie (stato_successivo, probabilità, ricompensa) per la coppia (stato, azione)
        for next_state, reward in self.table[key]:
            output[next_state][0] += 1  # Incrementa il conteggio delle visite
            output[next_state][1] += reward  # Somma le ricompense

        for next_state in output:
            visits, total_reward = output[next_state]
            output[next_state] = [visits / len(self.table[key]), total_reward / visits]  # Probabilità e ricompensa media

        return output


## Inserimento di alcune esperienze

Consideriamo un ambiente lineare con tre posizioni: `A → B → C`.

In [31]:
model = TableLookupModel()

# Simulazione di transizioni e aggiornamento del modello
for _ in range(10000):
    state = random.choice(STATES)
    action = random.choice(ACTIONS)
    transition_data = TRANSITION_PROBABILITIES[(state, action)]
    next_state = random.choices(transition_data["next_states"], weights=transition_data["probabilities"])[0]
    reward = random.gauss(transition_data["reward_mean"], transition_data["reward_variance"])
    
    model.update(state, action, next_state, reward)

## Interrogazione del modello

In [32]:
output = model.predict(STATES[0], ACTIONS[0])

print(f"Predizione per Stato: {STATES[0]}, Azione: {ACTIONS[0]}")
for next_state, (probability, mean_reward) in output.items():
    print(f"  Stato successivo: {next_state}, Probabilità: {probability:.2f}, Ricompensa media: {mean_reward:.2f}")

Predizione per Stato: A, Azione: left
  Stato successivo: B, Probabilità: 0.46, Ricompensa media: -0.89
  Stato successivo: A, Probabilità: 0.54, Ricompensa media: -0.99


In [33]:
# Calcolo dell'errore commesso dal modello rispetto alle transizioni reali
# Per ogni coppia (stato, azione) nota al modello viene fornito l'errore su probabilità e ricompensa media rispetto alle transizioni reali definite in TRANSITION_PROBABILITIES
errors = {}
for (state, action), data in TRANSITION_PROBABILITIES.items():
    try:
        model_output = model.predict(state, action)
    except KeyError:
        continue  # Se il modello non ha ancora visto questa coppia (stato, azione), salta

    real_next_states = data["next_states"]
    real_probabilities = data["probabilities"]
    real_reward_mean = data["reward_mean"]

    # Calcolo dell'errore sulle probabilità
    model_probabilities = [model_output.get(ns, [0.0, 0.0])[0] for ns in real_next_states]
    prob_error = sum(abs(mp - rp) for mp, rp in zip(model_probabilities, real_probabilities))

    # Calcolo dell'errore sulla ricompensa media
    model_reward_means = [model_output.get(ns, [0.0, 0.0])[1] for ns in real_next_states]
    reward_error = sum(abs(mr - real_reward_mean) for mr in model_reward_means)

    errors[(state, action)] = {
        "probability_error": prob_error,
        "reward_error": reward_error,
    }

print("\nErrori del modello rispetto alle transizioni reali:")
for (state, action), error_data in errors.items():
    print(f"Stato: {state}, Azione: {action}, Errore probabilità: {error_data['probability_error']:.2f}, Errore ricompensa: {error_data['reward_error']:.2f}")
    


Errori del modello rispetto alle transizioni reali:
Stato: A, Azione: left, Errore probabilità: 0.02, Errore ricompensa: 0.10
Stato: A, Azione: right, Errore probabilità: 0.01, Errore ricompensa: 0.10
Stato: B, Azione: left, Errore probabilità: 0.05, Errore ricompensa: 0.05
Stato: B, Azione: right, Errore probabilità: 0.02, Errore ricompensa: 0.02
Stato: C, Azione: left, Errore probabilità: 0.02, Errore ricompensa: 0.06
Stato: C, Azione: right, Errore probabilità: 0.02, Errore ricompensa: 0.12
Stato: D, Azione: left, Errore probabilità: 0.03, Errore ricompensa: 0.10
Stato: D, Azione: right, Errore probabilità: 0.06, Errore ricompensa: 0.02
